In [ ]:
from enum import Enum
from typing import NamedTuple

import numpy as np
import pandas as pd
import uproot

In [ ]:
class Benchmark(Enum):
    SM = 0
    W = 1
    NEG_W = 2
    WW = 3
    NEG_WW = 4


class EFTCoefficients(NamedTuple):
    cwl2: float
    cpwl2: float


# Map the benchmark names to their couplings (CWL2, CPWL2)
benchmarks_to_eftcoeffs = {
    Benchmark.SM: EFTCoefficients(0.0, 0.0),
    Benchmark.W: EFTCoefficients(15.2, 0.1),
    Benchmark.NEG_W: EFTCoefficients(-15.2, 0.2),
    Benchmark.WW: EFTCoefficients(0.3, 15.1),
    Benchmark.NEG_WW: EFTCoefficients(0.4, -15.3),
}

benchmarks_to_name = {
    Benchmark.SM: "sm",
    Benchmark.W: "w",
    Benchmark.NEG_W: "neg_w",
    Benchmark.WW: "ww",
    Benchmark.NEG_WW: "neg_ww",
}

In [ ]:
# load the data
f = uproot.open("~/Downloads/combined_samples-2.root")["Events"]
array = f.arrays(library="pd") # load the data as a pandas DataFrame

datasets = {
    benchmark: array[array["sampling_benchmark_id"] == benchmark.value]
    for benchmark in Benchmark
}

In [ ]:
from nsbi_common_utils.training import density_ratio_trainer


feature_list = ["pt_j1", "delta_phi_jj", "met"]

def phase_space_cuts(df: pd.DataFrame) -> pd.DataFrame:
    return df[(df["pt_j1"] < 500) & (df["met"] < 100)]


def prepare(
    datasets: dict[Benchmark, pd.DataFrame],
    benchmark: Benchmark,
    weight: str | None = None,
) -> tuple[density_ratio_trainer, pd.DataFrame]:
    """Prepare the dataset for training the density ratio estimator.

    Example:

        # Nominal training for WW benchmark
        trainer, df = prepare(datasets, Benchmark.WW, weight=None)

        # Training for a systematic variation, e.g. `weight_scale_mur_nuisance_param_0_up`
        trainer, df = prepare(datasets, Benchmark.WW, weight="weight_scale_mur_nuisance_param_0_up")

    """
    # assert benchmark != Benchmark.SM, "The benchmark should not be the SM, as it is used as the reference distribution."
    
    # prepare the dataset for training the density ratio estimator
    # Ref = pd.concat([
    #     phase_space_cuts(datasets[Benchmark.SM]), 
    #     phase_space_cuts(datasets[Benchmark.W]),
    # ], ignore_index=True)
    Ref = phase_space_cuts(datasets[Benchmark.SM])

    B = phase_space_cuts(datasets[benchmark])
    
    df = pd.concat([Ref, B], ignore_index=True)
    
    # if weight is none, use the weight column corresponding to the benchmark,
    # otherwise use the specified weight column e.g. for a systematic variation
    B_weight = f"weight_{benchmarks_to_name[benchmark]}"
    if weight is None:
        df["weights"] = np.where(
            df["sampling_benchmark_id"] == benchmark.value, df[B_weight], df["weight_sm"]
        )
    else:
        assert weight in df.columns, f"The weight column {weight} is not present in the dataset."
        scale_factors = df[weight] / df.weight_sm
        print(scale_factors.max())
        df["weights"] = np.where(
            df["sampling_benchmark_id"] == benchmark.value, df[B_weight] * scale_factors, df["weight_sm"]
        )
    
    df["train_labels"] = np.where(df["sampling_benchmark_id"] == benchmark.value, 1, 0).astype("int")
    df = df.astype('float32')


    Ref_name = "Ref"
    B_name = benchmarks_to_name[benchmark]
    output_name = f"{Ref_name}_vs_{B_name}" + f"__{weight}" if weight is not None else "__nominal"

    # initialize the density ratio trainer
    import re

    if weight is not None:
        name, dir = re.match(r"^weight_scale_(.+)_param_0_(.+)$", weight).groups()
        B_sample_name = f"{B_name} ({name} {dir})"
    else:
        B_sample_name = B_name + " (nominal)"

    trainer = density_ratio_trainer(
        dataset=df,
        weights=df["weights"],
        training_labels=df["train_labels"],
        features=feature_list,
        features_scaling=feature_list,
        sample_name=[Ref_name, B_sample_name],
        output_name=output_name,
        path_to_figures="plots/",
        path_to_models="models/",
    )
    return trainer, df

In [ ]:
# nominal training for all benchmarks
from typing import Any

nominal_trained_models: dict[Benchmark, Any] = {}

for benchmark in [Benchmark.WW, Benchmark.W]:
    trainer, df = prepare(datasets, benchmark, weight=None)

    trained_model = trainer.train(
        hidden_layers=4,
        neurons=1000,
        number_of_epochs=10,
        batch_size=1024,
        learning_rate=1e-3,
        scalerType="StandardScaler",
        ensemble_index=0,
    )

    trainer.make_reweighted_plots(feature_list, "log", 50)

    nominal_trained_models[benchmark] = trained_model

In [ ]:
trainer_syst, df_syst = prepare(datasets, Benchmark.WW, weight="weight_scale_muf_nuisance_param_0_up")
trainer_nom, df_nom = prepare(datasets, Benchmark.WW, weight=None)



np.sum(df_nom["weights"][df_nom["train_labels"] == 1]), np.sum(df_syst["weights"][df_syst["train_labels"] == 1])


In [ ]:
trainer_syst_, df_syst_ = prepare(datasets, Benchmark.WW, weight="weight_scale_muf_nuisance_param_0_up")

trained_model_ = trainer_syst_.train(
    hidden_layers=4,
    neurons=1000,
    number_of_epochs=10,
    batch_size=1024,
    learning_rate=1e-4,
    scalerType="StandardScaler",
    ensemble_index=0,
)

trainer_syst_.make_reweighted_plots(feature_list, "log", 50)

# print(trained_model_.summary())

In [ ]:
# make trainer for training with systematic variations, e.g. `weight_scale_muf_nuisance_param_0_up`
trainer_syst, df_syst = prepare(datasets, Benchmark.WW, weight="weight_scale_muf_nuisance_param_0_up")
# trainer_syst, df_syst = prepare(datasets, Benchmark.WW, weight="weight_scale_corr_nuisance_param_0_down")


# finetune the nominal model for the WW benchmark with the systematic variation `weight_scale_muf_nuisance_param_0_up` using LoRA finetuning
lora_rank = 16
lora_alpha = 2 * lora_rank

finetuned_model = trainer_syst.lora_finetune(
    model_to_finetune=nominal_trained_models[Benchmark.WW],
    lora_rank=lora_rank,
    lora_alpha=lora_alpha,
    learning_rate=1e-4,
    number_of_epochs=10,
    batch_size=1024,
    scalerType="StandardScaler",
    ensemble_index=0,
)

# print(finetuned_model.summary())

trainer_syst.make_reweighted_plots(feature_list, "log", 50)


In [ ]:
import nsbi_common_utils


path_to_trained_models = "models/"
ensemble_index = 0

path_to_saved_scaler = f"{path_to_trained_models}model_scaler{ensemble_index}.bin"
path_to_saved_model = f"{path_to_trained_models}model{ensemble_index}.onnx"

scaler, model_NN = nsbi_common_utils.training.load_trained_model(path_to_saved_model, path_to_saved_scaler)
score_pred = nsbi_common_utils.training.predict_with_model(datasets[Benchmark.SM][feature_list].astype('float32'), scaler, trainer_syst.model_NN)
ratio_pred = nsbi_common_utils.training.convert_score_to_ratio(score_pred)

In [ ]:
scale_factors = datasets[Benchmark.SM]["weight_scale_muf_nuisance_param_0_up"] / datasets[Benchmark.SM]["weight_sm"]
numerator = datasets[Benchmark.SM]["weight_ww"] * scale_factors
denominator = datasets[Benchmark.SM]["weight_sm"]
truth = numerator / denominator

idx = np.argwhere(np.isfinite(ratio_pred)).reshape(-1)
print(ratio_pred)
mse = np.mean((np.log(np.asarray(truth)[idx]) - np.log(ratio_pred[idx]))**2)
mse

In [ ]:
import nsbi_common_utils

results_from_scratch = {}

for dataset_size in [1000, 5000, 10000, 50000, 100000, 500000, 1000000]:
    ds = {k: v.sample(n=dataset_size, random_state=42) for k, v in datasets.items()}
    trainer_syst, df_syst = prepare(ds, Benchmark.WW, weight="weight_scale_muf_nuisance_param_0_up")

    # from sratch
    trained_model = trainer_syst.train(
        hidden_layers=4,
        neurons=1000,
        number_of_epochs=10,
        batch_size=1024,
        learning_rate=1e-4,
        scalerType="StandardScaler",
        ensemble_index=0,
    )

    path_to_trained_models = "models/"
    ensemble_index = 0

    path_to_saved_scaler = f"{path_to_trained_models}model_scaler{ensemble_index}.bin"
    path_to_saved_model = f"{path_to_trained_models}model{ensemble_index}.onnx"

    scaler, model_NN = nsbi_common_utils.training.load_trained_model(path_to_saved_model, path_to_saved_scaler)
    score_pred = nsbi_common_utils.training.predict_with_model(ds[Benchmark.SM][feature_list].astype('float32'), scaler, trainer_syst.model_NN)
    ratio_pred = nsbi_common_utils.training.convert_score_to_ratio(score_pred)
    
    truth = ds[Benchmark.SM]["weight_ww"] / ds[Benchmark.SM]["weight_sm"]

    mse = np.mean((np.log(truth) - np.log(ratio_pred))**2)
    
    results_from_scratch[dataset_size] = mse

In [ ]:
import nsbi_common_utils

results_lora = {}

for dataset_size in [1000, 5000, 10000, 50000, 100000, 500000, 1000000]:
    ds = {k: v.sample(n=dataset_size, random_state=42) for k, v in datasets.items()}
    trainer_syst, df_syst = prepare(ds, Benchmark.WW, weight="weight_scale_muf_nuisance_param_0_up")


    # finetune the nominal model for the WW benchmark with the systematic variation `weight_scale_muf_nuisance_param_0_up` using LoRA finetuning
    lora_rank = 16
    lora_alpha = 2 * lora_rank

    finetuned_model = trainer_syst.lora_finetune(
        model_to_finetune=nominal_trained_models[Benchmark.WW],
        lora_rank=lora_rank,
        lora_alpha=lora_alpha,
        learning_rate=1e-4,
        number_of_epochs=10,
        batch_size=1024,
        scalerType="StandardScaler",
        ensemble_index=0,
    )

    path_to_trained_models = "models/"
    ensemble_index = 0

    path_to_saved_scaler = f"{path_to_trained_models}model_scaler{ensemble_index}.bin"
    path_to_saved_model = f"{path_to_trained_models}model{ensemble_index}.onnx"

    scaler, model_NN = nsbi_common_utils.training.load_trained_model(path_to_saved_model, path_to_saved_scaler)
    score_pred = nsbi_common_utils.training.predict_with_model(ds[Benchmark.SM][feature_list].astype('float32'), scaler, trainer_syst.model_NN)
    ratio_pred = nsbi_common_utils.training.convert_score_to_ratio(score_pred)
    
    truth = ds[Benchmark.SM]["weight_ww"] / ds[Benchmark.SM]["weight_sm"]

    mse = np.mean((np.log(truth) - np.log(ratio_pred))**2)
    
    results_lora[dataset_size] = mse

In [ ]:
results

In [ ]:
truth = datasets[Benchmark.SM]["weight_ww"] / datasets[Benchmark.SM]["weight_sm"]

mse = np.mean((np.log(truth) - np.log(ratio_pred))**2)
mse